Standardise and align the validated datasets from Notebook 10A into a consistent analytical structure, ready for downstream modelling and insight generation.

Input loading - Load all required datasets and configuration files for harmonisation

In [110]:
# Imports and display
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [111]:
# Project path
PROJECT_DIR = Path.cwd()
print(f"Current working directory: {PROJECT_DIR}")

Current working directory: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities


In [112]:
# Input files from Notebook 10A and prior processed outputs
INPUT_FILES = {
    "fingertips_selected_indicators": PROJECT_DIR / "_processed_fingertips_nwl_selected_indicators.parquet",
    "gla_his_borough_latest_and_trend": PROJECT_DIR / "_processed_gla_his_borough_latest_and_trend.parquet",
    "census2021_ethnicity_nwl_analytical": PROJECT_DIR / "_processed_census2021_ethnicity_nwl_analytical.parquet",
    "ethnicity_facts_gp_satisfaction": PROJECT_DIR / "_processed_ethnicity_facts_gp_satisfaction.parquet",
    "gpps_survey": PROJECT_DIR / "_processed_gpps_survey.parquet",
    "gpps_ethnicity_distribution": PROJECT_DIR / "_processed_gpps_ethnicity_distribution.parquet",
    "ons_birth_characteristics_national": PROJECT_DIR / "_processed_ons_birth_characteristics_national.parquet",
    "nwl_boroughs": PROJECT_DIR / "_config_nwl_boroughs.csv",
    "ethnicity_mapping": PROJECT_DIR / "_config_ethnicity_mapping.csv",
}

In [113]:
# File existence check
file_check_rows = []

for file_name, file_path in INPUT_FILES.items():
    file_check_rows.append(
        {
            "file_name": file_name,
            "path": str(file_path),
            "exists": file_path.exists(),
        }
    )

file_check_df = pd.DataFrame(file_check_rows).sort_values("file_name").reset_index(drop=True)
file_check_df

,file_name,path,exists
0,census2021_ethnicity_nwl_analytical,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
1,ethnicity_facts_gp_satisfaction,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
2,ethnicity_mapping,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_config_e...,True
3,fingertips_selected_indicators,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
4,gla_his_borough_latest_and_trend,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
5,gpps_ethnicity_distribution,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
6,gpps_survey,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
7,nwl_boroughs,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_config_n...,True
8,ons_birth_characteristics_national,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True


In [114]:
# Load inputs
def load_input_files(input_files: dict) -> tuple[dict, pd.DataFrame]:
    loaded_files = {}
    load_rows = []

    for file_name, file_path in input_files.items():
        load_row = {
            "file_name": file_name,
            "file_type": file_path.suffix.lower(),
            "loaded": False,
            "rows": None,
            "columns": None,
            "error_type": None,
            "error_message": None,
        }

        try:
            if file_path.suffix.lower() == ".parquet":
                dataframe = pd.read_parquet(file_path)
            elif file_path.suffix.lower() == ".csv":
                dataframe = pd.read_csv(file_path)
            else:
                raise ValueError(f"Unsupported file type: {file_path.suffix}")

            loaded_files[file_name] = dataframe
            load_row["loaded"] = True
            load_row["rows"] = int(dataframe.shape[0])
            load_row["columns"] = int(dataframe.shape[1])

        except Exception as error:
            load_row["error_type"] = type(error).__name__
            load_row["error_message"] = str(error)

        load_rows.append(load_row)

    load_audit = pd.DataFrame(load_rows).sort_values("file_name").reset_index(drop=True)
    return loaded_files, load_audit

In [115]:
# Run load audit
loaded_inputs, load_audit = load_input_files(INPUT_FILES)

print(f"Loaded {load_audit['loaded'].sum()} of {len(INPUT_FILES)} input files.")
load_audit

Loaded 9 of 9 input files.


,file_name,file_type,loaded,rows,columns,error_type,error_message
0,census2021_ethnicity_nwl_analytical,.parquet,True,64,6,None,None
1,ethnicity_facts_gp_satisfaction,.parquet,True,21,11,None,None
2,ethnicity_mapping,.csv,True,8,2,None,None
3,fingertips_selected_indicators,.parquet,True,5118,28,None,None
4,gla_his_borough_latest_and_trend,.parquet,True,3611,14,None,None
5,gpps_ethnicity_distribution,.parquet,True,252,12,None,None
6,gpps_survey,.parquet,True,630,11,None,None
7,nwl_boroughs,.csv,True,8,2,None,None
8,ons_birth_characteristics_national,.parquet,True,995,13,None,None


In [116]:
# Assign loaded inputs to working dataframes
fingertips_df = loaded_inputs["fingertips_selected_indicators"].copy()
gla_df = loaded_inputs["gla_his_borough_latest_and_trend"].copy()
census_df = loaded_inputs["census2021_ethnicity_nwl_analytical"].copy()
eff_df = loaded_inputs["ethnicity_facts_gp_satisfaction"].copy()
gpps_survey_df = loaded_inputs["gpps_survey"].copy()
gpps_ethnicity_df = loaded_inputs["gpps_ethnicity_distribution"].copy()
ons_births_df = loaded_inputs["ons_birth_characteristics_national"].copy()

nwl_boroughs_df = loaded_inputs["nwl_boroughs"].copy()
ethnicity_mapping_df = loaded_inputs["ethnicity_mapping"].copy()

print("Working dataframes created.")

Working dataframes created.


Initial inspection - Check borough and ethnicity structures before applying harmonisation rules.


In [117]:
# inspect borough and ethnicity config files
print("NWL boroughs config")
display(nwl_boroughs_df)

print("\nEthnicity mapping config")
display(ethnicity_mapping_df)

NWL boroughs config


,borough,gss_code
0,Brent,E09000005
1,Ealing,E09000009
2,Hammersmith and Fulham,E09000013
3,Harrow,E09000015
4,Hillingdon,E09000017
5,Hounslow,E09000018
6,Kensington and Chelsea,E09000020
7,Westminster,E09000033



Ethnicity mapping config


,raw_ethnicity,standard_ethnicity
0,Black African,Black African
1,African,Black African
2,Black Caribbean,Black Caribbean
3,Caribbean,Black Caribbean
4,White British,White British
5,White: English/Welsh/Scottish/Northern Irish/British,White British
6,All,All
7,Total,All


In [118]:
# Inspect borough values across key datasets
print("NWL borough list from config:")
print(sorted(nwl_boroughs_df["borough"].dropna().astype(str).unique().tolist()))
print()

if "Area Name" in fingertips_df.columns:
    print("Fingertips borough sample:")
    print(sorted(fingertips_df["Area Name"].dropna().astype(str).unique().tolist())[:20])
    print()

if "borough" in gla_df.columns:
    print("GLA borough sample:")
    print(sorted(gla_df["borough"].dropna().astype(str).unique().tolist())[:20])
    print()

if "geography_name" in census_df.columns:
    print("Census borough sample:")
    print(sorted(census_df["geography_name"].dropna().astype(str).unique().tolist()))

NWL borough list from config:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']

Fingertips borough sample:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']

GLA borough sample:
['Area Name', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'City of London', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington']

Census borough sample:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']


In [119]:
# Inspect ethnicity values
if "analytical_ethnicity" in census_df.columns:
    print("Census analytical_ethnicity values:")
    print(sorted(census_df["analytical_ethnicity"].dropna().astype(str).unique().tolist()))
    print()

if "standard_ethnicity" in eff_df.columns:
    print("Ethnicity Facts standard_ethnicity values:")
    print(sorted(eff_df["standard_ethnicity"].dropna().astype(str).unique().tolist()))
    print()

if "ethnicity_standard" in gpps_ethnicity_df.columns:
    print("GPPS ethnicity_standard values:")
    print(sorted(gpps_ethnicity_df["ethnicity_standard"].dropna().astype(str).unique().tolist()))
    print()

if "ethnicity_standard" in ons_births_df.columns:
    print("ONS births ethnicity_standard values:")
    print(sorted(ons_births_df["ethnicity_standard"].dropna().astype(str).unique().tolist()))

Census analytical_ethnicity values:
['Asian', 'Black African', 'Black Caribbean', 'Black Other', 'Mixed', 'Other', 'White British', 'White Other']

Ethnicity Facts standard_ethnicity values:
['All', 'Any other ethnic background', 'Arab', 'Asian other', 'Bangladeshi', 'Black African', 'Black Caribbean', 'Black other', 'Chinese', 'Gypsy or Irish traveller', 'Indian', 'Irish', 'Mixed other', 'Mixed white and Asian', 'Mixed white and black African', 'Mixed white and black Caribbean', 'Pakistani', 'Roma', 'Unknown', 'White British', 'White other']

GPPS ethnicity_standard values:
['Asian', 'Black', 'Mixed', 'Not stated', 'Other', 'White']

ONS births ethnicity_standard values:
['All', 'Any other Asian background', 'Any other Black background', 'Any other White background', 'Any other ethnic group', 'Asian', 'Bangladeshi', 'Black', 'Black African', 'Black Caribbean', 'Indian', 'Mixed/multiple', 'Not stated', 'Pakistani', 'White', 'White British', 'White Other']


The main structural differences are now identified, so harmonisation rules can be applied consistently.

We currently have 4 incompatible ethnicity systems that needs to be brought into a project-level standard ethnicity grouping

Decision for this project - 

We will Use a 6-group standard (aligned to GPPS + policy usability):

White
Asian
Black
Mixed
Other
Not stated

This is the only level that:

exists across datasets (directly or via aggregation)
supports modelling
is interpretable for stakeholders

Based on the above dataset audits the next steps are:

We now:

- fix GLA (carry forward cleaning from 10A)
- standardise borough naming
- THEN build ethnicity harmonisation

In [120]:
# Clean GLA dataset

gla_df = gla_df.copy()

gla_df = gla_df[
    gla_df["borough"].notna()
    & (gla_df["borough"].astype(str).str.strip() != "")
    & (gla_df["borough"] != "Area Name")
]

cols_to_drop = ["", "unnamed", "unnamed__2", "Negative error bar", "Positive error bar"]
cols_to_drop = [col for col in cols_to_drop if col in gla_df.columns]

gla_df = gla_df.drop(columns=cols_to_drop)

print("GLA cleaned shape:", gla_df.shape)

GLA cleaned shape: (3609, 9)


In [121]:
# Enforce NWL borough filter across datasets

nwl_boroughs = set(nwl_boroughs_df["borough"])

# Fingertips
fingertips_df = fingertips_df[
    fingertips_df["Area Name"].isin(nwl_boroughs)
].copy()

# GLA
gla_df = gla_df[
    gla_df["borough"].isin(nwl_boroughs)
].copy()

# Census
census_df = census_df[
    census_df["geography_name"].isin(nwl_boroughs)
].copy()

print("Filtered dataset sizes:")
print("Fingertips:", fingertips_df.shape)
print("GLA:", gla_df.shape)
print("Census:", census_df.shape)

Filtered dataset sizes:
Fingertips: (5118, 28)
GLA: (888, 9)
Census: (64, 6)


In [122]:
# Quick check 
gla_df["borough"].unique()

<ArrowStringArray>
['Kensington and Chelsea', 'Brent', 'Hammersmith and Fulham', 'Hillingdon', 'Ealing', 'Harrow', 'Westminster', 'Hounslow']
Length: 8, dtype: string

Now we move to the core step: ethnicity harmonisation.

We create a single standard ethnicity field:

White / Asian / Black / Mixed / Other / Not stated

This will be applied across:

Census
GPPS
ONS
Ethnicity Facts

In [123]:
# Standard ethnicity mapping
def map_to_standard_ethnicity(value: str) -> str:
    if pd.isna(value):
        return "Not stated"

    value = str(value).strip().lower()

    # White
    if "white" in value:
        return "White"

    # Asian
    if any(x in value for x in ["asian", "indian", "pakistani", "bangladeshi", "chinese"]):
        return "Asian"

    # Black
    if any(x in value for x in ["black", "african", "caribbean"]):
        return "Black"

    # Mixed
    if "mixed" in value:
        return "Mixed"

    # Not stated / unknown
    if any(x in value for x in ["not stated", "unknown"]):
        return "Not stated"

    # Other fallback
    return "Other"


In [124]:
# Apply mapping to all relevant datasets
# Census
census_df["ethnicity_standard"] = census_df["analytical_ethnicity"].apply(map_to_standard_ethnicity)

# Ethnicity Facts
eff_df["ethnicity_standardised"] = eff_df["standard_ethnicity"].apply(map_to_standard_ethnicity)

# GPPS (already standard but normalise anyway)
gpps_ethnicity_df["ethnicity_standardised"] = gpps_ethnicity_df["ethnicity_standard"].apply(map_to_standard_ethnicity)

# ONS births
ons_births_df["ethnicity_standardised"] = ons_births_df["ethnicity_standard"].apply(map_to_standard_ethnicity)

print("Ethnicity mapping applied.")

Ethnicity mapping applied.


In [125]:
# Validate mapping results
print("Census mapped:")
print(sorted(census_df["ethnicity_standard"].unique()))
print()

print("Ethnicity Facts mapped:")
print(sorted(eff_df["ethnicity_standardised"].unique()))
print()

print("GPPS mapped:")
print(sorted(gpps_ethnicity_df["ethnicity_standardised"].unique()))
print()

print("ONS mapped:")
print(sorted(ons_births_df["ethnicity_standardised"].unique()))

Census mapped:
['Asian', 'Black', 'Mixed', 'Other', 'White']

Ethnicity Facts mapped:
['Asian', 'Black', 'Mixed', 'Not stated', 'Other', 'White']

GPPS mapped:
['Asian', 'Black', 'Mixed', 'Not stated', 'Other', 'White']

ONS mapped:
['Asian', 'Black', 'Mixed', 'Not stated', 'Other', 'White']


Next we start building the first harmonised table: population layer from Census. Then we will create consistent analytical tables for population, experience, outcomes, determinants, and national disparities.

In [126]:
# Harmonise Census population
census_population_harmonised = (
    census_df.rename(
        columns={
            "geography_code": "area_code",
            "geography_name": "area_name",
            "population": "value",
        }
    )
    .assign(
        table="Census2021",
        source="Census 2021",
        year=2021,
        geography="England",
        area_geography="Borough",
        category="Population",
        metric="population_count",
        measure_raw="population",
    )[
        [
            "table",
            "source",
            "year",
            "geography",
            "area_geography",
            "area_code",
            "area_name",
            "category",
            "metric",
            "analytical_ethnicity",
            "ethnicity_standard",
            "measure_raw",
            "value",
            "borough_total",
            "proportion",
        ]
    ]
    .sort_values(["area_name", "ethnicity_standard", "analytical_ethnicity"])
    .reset_index(drop=True)
)

census_population_harmonised.head(10)

,table,source,year,geography,area_geography,area_code,area_name,category,metric,analytical_ethnicity,ethnicity_standard,measure_raw,value,borough_total,proportion
0,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,Asian,Asian,population,111515,339821,0.328158
1,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,Black African,Black,population,31070,339821,0.091430
2,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,Black Caribbean,Black,population,21258,339821,0.062556
3,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,Black Other,Black,population,7167,339821,0.021091
4,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,Mixed,Mixed,population,17249,339821,0.050759
5,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,Other,Other,population,33861,339821,0.099644
6,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,White British,White,population,51611,339821,0.151877
7,Census2021,Census 2021,2021,England,Borough,E09000005,Brent,Population,population_count,White Other,White,population,66090,339821,0.194485
8,Census2021,Census 2021,2021,England,Borough,E09000009,Ealing,Population,population_count,Asian,Asian,population,111241,367116,0.303013
9,Census2021,Census 2021,2021,England,Borough,E09000009,Ealing,Population,population_count,Black African,Black,population,22578,367116,0.061501


In [127]:
# Validate Census population
print("Shape:", census_population_harmonised.shape)
print()

print("Columns:")
print(census_population_harmonised.columns.tolist())
print()

print("Ethnicity standard values:")
print(sorted(census_population_harmonised["ethnicity_standard"].dropna().unique().tolist()))
print()

print("Area names:")
print(sorted(census_population_harmonised["area_name"].dropna().unique().tolist()))


Shape: (64, 15)

Columns:
['table', 'source', 'year', 'geography', 'area_geography', 'area_code', 'area_name', 'category', 'metric', 'analytical_ethnicity', 'ethnicity_standard', 'measure_raw', 'value', 'borough_total', 'proportion']

Ethnicity standard values:
['Asian', 'Black', 'Mixed', 'Other', 'White']

Area names:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']


Next: we will build the GPPS ethnicity composition layer in the same harmonised structure.

In [128]:
# Harmonise GPPS ethnicity distribution
gpps_ethnicity_harmonised = (
    gpps_ethnicity_df.copy()
    .drop(columns=["ethnicity_standard"], errors="ignore")
    .rename(
        columns={
            "ethnicity_standardised": "ethnicity_standard",
        }
    )
    .assign(
        category="Population composition",
        metric="population_distribution",
    )[
        [
            "table",
            "source",
            "year",
            "geography",
            "area_geography",
            "area_code",
            "area_name",
            "category",
            "metric",
            "ethnicity_raw",
            "ethnicity_standard",
            "measure_raw",
            "value",
        ]
    ]
    .sort_values(["area_name", "ethnicity_standard", "measure_raw"])
    .reset_index(drop=True)
)

gpps_ethnicity_harmonised.head(10)

,table,source,year,geography,area_geography,area_code,area_name,category,metric,ethnicity_raw,ethnicity_standard,measure_raw,value
0,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Population composition,population_distribution,3,Asian,dv_ethnicityband_3.pct,0.060757
1,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Population composition,population_distribution,4,Black,dv_ethnicityband_4.pct,0.018793
2,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Population composition,population_distribution,2,Mixed,dv_ethnicityband_2.pct,0.015130
3,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Population composition,population_distribution,6,Not stated,dv_ethnicityband_6.pct,0.009201
4,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Population composition,population_distribution,5,Other,dv_ethnicityband_5.pct,0.005492
5,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Population composition,population_distribution,1,White,dv_ethnicityband_1.pct,0.890627
6,GPPS_ICS,GP Patient Survey,2025,England,ICS,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRATED CARE SYSTEM",Population composition,population_distribution,3,Asian,dv_ethnicityband_3.pct,0.170797
7,GPPS_ICS,GP Patient Survey,2025,England,ICS,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRATED CARE SYSTEM",Population composition,population_distribution,4,Black,dv_ethnicityband_4.pct,0.099631
8,GPPS_ICS,GP Patient Survey,2025,England,ICS,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRATED CARE SYSTEM",Population composition,population_distribution,2,Mixed,dv_ethnicityband_2.pct,0.022057
9,GPPS_ICS,GP Patient Survey,2025,England,ICS,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRATED CARE SYSTEM",Population composition,population_distribution,6,Not stated,dv_ethnicityband_6.pct,0.015995


In [129]:
# Validate GPPS ethnicity distribution

print("Shape:", gpps_ethnicity_harmonised.shape)
print()

print("Columns:")
print(gpps_ethnicity_harmonised.columns.tolist())
print()

print("Ethnicity standard values:")
print(sorted(gpps_ethnicity_harmonised["ethnicity_standard"].dropna().unique().tolist()))
print()

print("Area geography values:")
print(sorted(gpps_ethnicity_harmonised["area_geography"].dropna().unique().tolist()))
print()

print("Metric values:")
print(sorted(gpps_ethnicity_harmonised["metric"].dropna().unique().tolist()))

Shape: (252, 13)

Columns:
['table', 'source', 'year', 'geography', 'area_geography', 'area_code', 'area_name', 'category', 'metric', 'ethnicity_raw', 'ethnicity_standard', 'measure_raw', 'value']

Ethnicity standard values:
['Asian', 'Black', 'Mixed', 'Not stated', 'Other', 'White']

Area geography values:
['ICS']

Metric values:
['population_distribution']


Next: we will build the GPPS experience layer.

In [130]:
# Harmonise GPPS survey experience
gpps_experience_harmonised = (
    gpps_survey_df.copy()
    .rename(columns={"category": "response_category"})
    .assign(
        category="Service experience",
    )[
        [
            "table",
            "source",
            "year",
            "geography",
            "area_geography",
            "area_code",
            "area_name",
            "category",
            "metric",
            "response_category",
            "measure_raw",
            "value",
        ]
    ]
    .sort_values(["area_name", "metric", "response_category", "measure_raw"])
    .reset_index(drop=True)
)

gpps_experience_harmonised.head(10)

,table,source,year,geography,area_geography,area_code,area_name,category,metric,response_category,measure_raw,value
0,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,gpcontactoverall,1,gpcontactoverall_1.pct,0.450732
1,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,gpcontactoverall,2,gpcontactoverall_2.pct,0.297158
2,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,gpcontactoverall,3,gpcontactoverall_3.pct,0.122938
3,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,gpcontactoverall,4,gpcontactoverall_4.pct,0.072810
4,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,gpcontactoverall,5,gpcontactoverall_5.pct,0.056361
5,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,healthconfidence,1,healthconfidence_1.pct,0.276817
6,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,healthconfidence,2,healthconfidence_2.pct,0.548814
7,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,healthconfidence,3,healthconfidence_3.pct,0.139188
8,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,healthconfidence,4,healthconfidence_4.pct,0.035181
9,GPPS_ICS,GP Patient Survey,2025,England,ICS,QOX,"BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE SYSTEM",Service experience,healthconfidence,5,healthconfidence_5.pct,0.028422


In [131]:
# Validate GPPS survey experience
print("Shape:", gpps_experience_harmonised.shape)
print()

print("Columns:")
print(gpps_experience_harmonised.columns.tolist())
print()

print("Area geography values:")
print(sorted(gpps_experience_harmonised["area_geography"].dropna().unique().tolist()))
print()

print("Metric values:")
print(sorted(gpps_experience_harmonised["metric"].dropna().unique().tolist()))
print()

print("Response category sample:")
print(sorted(gpps_experience_harmonised["response_category"].dropna().astype(str).unique().tolist())[:10])

Shape: (630, 12)

Columns:
['table', 'source', 'year', 'geography', 'area_geography', 'area_code', 'area_name', 'category', 'metric', 'response_category', 'measure_raw', 'value']

Area geography values:
['ICS']

Metric values:
['gpcontactoverall', 'healthconfidence', 'overallexp']

Response category sample:
['1', '2', '3', '4', '5']


Next: we will build the ONS national ethnicity disparity layer.

In [132]:
# Harmonise ONS births
ons_births_harmonised = (
    ons_births_df.copy()
    .drop(columns=["ethnicity_standard"], errors="ignore")
    .rename(columns={"ethnicity_standardised": "ethnicity_standard"})
    .assign(
        category_group="Birth outcomes",
    )[
        [
            "table",
            "source",
            "year",
            "geography",
            "area_geography",
            "area_code",
            "area_name",
            "category_group",
            "category",
            "metric",
            "ethnicity_raw",
            "ethnicity_standard",
            "measure_raw",
            "value",
        ]
    ]
    .sort_values(["year", "metric", "ethnicity_standard", "category"])
    .reset_index(drop=True)
)

ons_births_harmonised.head(10)

,table,source,year,geography,area_geography,area_code,area_name,category_group,category,metric,ethnicity_raw,ethnicity_standard,measure_raw,value
0,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Bangladeshi,Asian,Number of live births Bangladeshi,9249.0
1,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Indian,Asian,Number of live births Indian,18106.0
2,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Pakistani,Asian,Number of live births Pakistani,26108.0
3,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Any other Asian background,Asian,Number of live births Any other Asian background,12587.0
4,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Black African,Black,Number of live births Black African,22399.0
5,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Black Caribbean,Black,Number of live births Black Caribbean,7427.0
6,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Any other Black background,Black,Number of live births Any other Black background,5635.0
7,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Mixed/multiple,Mixed,Number of live births Mixed/ multiple,26254.0
8,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Not stated,Not stated,Number of live births Not stated,62386.0
9,Table_20,ONS Birth Characteristics,2007,England and Wales,Country,K04000001,ENGLAND AND WALES,Birth outcomes,All,number_live_births,Total,Other,Number of live births Total,689120.0


In [133]:
# Validate ONS births
print("Shape:", ons_births_harmonised.shape)
print()

print("Columns:")
print(ons_births_harmonised.columns.tolist())
print()

print("Ethnicity standard values:")
print(sorted(ons_births_harmonised["ethnicity_standard"].dropna().unique().tolist()))
print()

print("Area geography values:")
print(sorted(ons_births_harmonised["area_geography"].dropna().unique().tolist()))
print()

print("Metric values:")
print(sorted(ons_births_harmonised["metric"].dropna().unique().tolist()))

Shape: (995, 14)

Columns:
['table', 'source', 'year', 'geography', 'area_geography', 'area_code', 'area_name', 'category_group', 'category', 'metric', 'ethnicity_raw', 'ethnicity_standard', 'measure_raw', 'value']

Ethnicity standard values:
['Asian', 'Black', 'Mixed', 'Not stated', 'Other', 'White']

Area geography values:
['Country']

Metric values:
['live_births', 'number_live_births', 'number_stillbirths', 'stillbirth_rate', 'total_live_births', 'total_stillbirths']


Now we complete the core integration layer by bringing in Fingertips (outcomes).

In [134]:
# Harmonise Fingertips outcomes
fingertips_harmonised = (
    fingertips_df.copy()
    .loc[lambda df: df["Area Type"].astype(str).str.strip() == "UA"]
    .rename(
        columns={
            "Area Code": "area_code",
            "Area Name": "area_name",
            "Area Type": "area_type",
            "Time period": "time_period",
            "Value": "value",
            "Category": "category",
            "Indicator Name": "metric",
        }
    )
    .assign(
        table="Fingertips",
        source="OHID Fingertips",
        geography="England",
        area_geography="Borough",
        measure_raw="value",
    )[
        [
            "table",
            "source",
            "geography",
            "area_geography",
            "area_code",
            "area_name",
            "area_type",
            "time_period",
            "category",
            "metric",
            "measure_raw",
            "value",
        ]
    ]
    .sort_values(["area_name", "metric", "time_period"])
    .reset_index(drop=True)
)

fingertips_harmonised.head(10)

,table,source,geography,area_geography,area_code,area_name,area_type,time_period,category,metric,measure_raw,value
0,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2014/15,<NA>,% reporting a long term MSK problem who also report depression or anxiety,value,25.987533
1,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2015/16,<NA>,% reporting a long term MSK problem who also report depression or anxiety,value,26.395159
2,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2016/17,<NA>,% reporting a long term MSK problem who also report depression or anxiety,value,23.705820
3,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2020,<NA>,% reporting a long-term mental health problem,value,6.391032
4,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2020,<NA>,% reporting cancer in the last 5 years,value,2.793530
5,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2014/15,<NA>,% reporting depression or anxiety,value,11.755264
6,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2015/16,<NA>,% reporting depression or anxiety,value,10.994544
7,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2016/17,<NA>,% reporting depression or anxiety,value,11.034460
8,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2020,<NA>,% with caring responsibility,value,13.724738
9,Fingertips,OHID Fingertips,England,Borough,E09000005,Brent,UA,2013/14,<NA>,Abdominal Aortic Aneurysm Screening Coverage,value,67.221298


In [135]:
# Validate Fingertips outcomes
print("Shape:", fingertips_harmonised.shape)
print()

print("Columns:")
print(fingertips_harmonised.columns.tolist())
print()

print("Area types:")
print(sorted(fingertips_harmonised["area_type"].dropna().unique().tolist()))
print()

print("Area names:")
print(sorted(fingertips_harmonised["area_name"].dropna().unique().tolist()))
print()

print("Area codes:")
print(sorted(fingertips_harmonised["area_code"].dropna().unique().tolist()))

Shape: (5112, 12)

Columns:
['table', 'source', 'geography', 'area_geography', 'area_code', 'area_name', 'area_type', 'time_period', 'category', 'metric', 'measure_raw', 'value']

Area types:
['UA']

Area names:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']

Area codes:
['E09000005', 'E09000009', 'E09000013', 'E09000015', 'E09000017', 'E09000018', 'E09000020', 'E09000033']


At this point we have 

-  Census → population
-  GPPS → ethnicity composition
-  GPPS → experience
-  ONS → national disparities
-  Fingertips → local outcomes

Next, we will harmonise GLA → determinants

In [136]:
# Harmonise GLA determinants
gla_harmonised = (
    gla_df.copy()
    .rename(
        columns={
            "borough": "area_name",
            "time_period": "time_period",
            "value": "value",
            "indicator_group": "metric",
        }
    )
    .assign(
        table="GLA HIS",
        source="GLA Historical Indicators",
        geography="England",
        area_geography="Borough",
        category="Wider determinants",
        measure_raw="value",
    )[
        [
            "table",
            "source",
            "geography",
            "area_geography",
            "area_name",
            "time_period",
            "category",
            "metric",
            "measure_raw",
            "value",
        ]
    ]
    .sort_values(["area_name", "metric", "time_period"])
    .reset_index(drop=True)
)

gla_harmonised.head(10)

,table,source,geography,area_geography,area_name,time_period,category,metric,measure_raw,value
0,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2011 - 13,Wider determinants,1. HLE male,value,62.64
1,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2011 - 13,Wider determinants,1. HLE male,value,62.64
2,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2012 - 14,Wider determinants,1. HLE male,value,64.33
3,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2012 - 14,Wider determinants,1. HLE male,value,64.33
4,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2013 - 15,Wider determinants,1. HLE male,value,63.71
5,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2013 - 15,Wider determinants,1. HLE male,value,63.71
6,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2014 - 16,Wider determinants,1. HLE male,value,64.32
7,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2014 - 16,Wider determinants,1. HLE male,value,64.32
8,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2015 - 17,Wider determinants,1. HLE male,value,63.34
9,GLA HIS,GLA Historical Indicators,England,Borough,Brent,2015 - 17,Wider determinants,1. HLE male,value,63.34


In [137]:
# Validate GLA determinants
print("Shape:", gla_harmonised.shape)
print()

print("Columns:")
print(gla_harmonised.columns.tolist())
print()

print("Area names:")
print(sorted(gla_harmonised["area_name"].dropna().unique().tolist()))
print()

print("Metric sample:")
print(gla_harmonised["metric"].dropna().unique()[:10])

Shape: (888, 10)

Columns:
['table', 'source', 'geography', 'area_geography', 'area_name', 'time_period', 'category', 'metric', 'measure_raw', 'value']

Area names:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']

Metric sample:
<ArrowStringArray>
[                 '1. HLE male',       '11. HIV late diagnosis',       '14. Smoking prevalence',                '2. HLE female',
          '3. Low Birth Weight',      '4. School readiness FSM',   '5. Excess weight age 10-11',   '6. Excess weight age 04-05',
              '7. Suicide rate', '8. Mortality caused by PM2.5']
Length: 10, dtype: string


Issue

gla_harmonised has no area_code, while Census and Fingertips do.
We should add borough codes from nwl_boroughs_df now.

In [138]:
# Add area codes to GLA
gla_harmonised = gla_harmonised.merge(
    nwl_boroughs_df.rename(columns={"borough": "area_name", "gss_code": "area_code"}),
    on="area_name",
    how="left"
)

# Reorder columns
gla_harmonised = gla_harmonised[
    [
        "table",
        "source",
        "geography",
        "area_geography",
        "area_code",
        "area_name",
        "time_period",
        "category",
        "metric",
        "measure_raw",
        "value",
    ]
].copy()

gla_harmonised.head(10)

,table,source,geography,area_geography,area_code,area_name,time_period,category,metric,measure_raw,value
0,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2011 - 13,Wider determinants,1. HLE male,value,62.64
1,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2011 - 13,Wider determinants,1. HLE male,value,62.64
2,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2012 - 14,Wider determinants,1. HLE male,value,64.33
3,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2012 - 14,Wider determinants,1. HLE male,value,64.33
4,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2013 - 15,Wider determinants,1. HLE male,value,63.71
5,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2013 - 15,Wider determinants,1. HLE male,value,63.71
6,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2014 - 16,Wider determinants,1. HLE male,value,64.32
7,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2014 - 16,Wider determinants,1. HLE male,value,64.32
8,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2015 - 17,Wider determinants,1. HLE male,value,63.34
9,GLA HIS,GLA Historical Indicators,England,Borough,E09000005,Brent,2015 - 17,Wider determinants,1. HLE male,value,63.34


In [139]:
# Validate GLA area codes
print("Shape:", gla_harmonised.shape)
print()

print("Missing area_code rows:", gla_harmonised["area_code"].isna().sum())
print()

print("Area code sample:")
print(sorted(gla_harmonised["area_code"].dropna().unique().tolist()))

Shape: (888, 11)

Missing area_code rows: 0

Area code sample:
['E09000005', 'E09000009', 'E09000013', 'E09000015', 'E09000017', 'E09000018', 'E09000020', 'E09000033 ']


One area_code has a trailing space.

'E09000033 '

That should be cleaned now, before exports.

In [140]:
# Clean area codes
def clean_code_column(dataframe: pd.DataFrame, column_name: str) -> pd.DataFrame:
    dataframe = dataframe.copy()

    if column_name in dataframe.columns:
        dataframe[column_name] = (
            dataframe[column_name]
            .astype("string")
            .str.strip()
        )

    return dataframe


census_population_harmonised = clean_code_column(census_population_harmonised, "area_code")
gpps_ethnicity_harmonised = clean_code_column(gpps_ethnicity_harmonised, "area_code")
gpps_experience_harmonised = clean_code_column(gpps_experience_harmonised, "area_code")
ons_births_harmonised = clean_code_column(ons_births_harmonised, "area_code")
fingertips_harmonised = clean_code_column(fingertips_harmonised, "area_code")
gla_harmonised = clean_code_column(gla_harmonised, "area_code")

print("Area code cleaning complete.")

Area code cleaning complete.


In [141]:
# Re-check area codes
print("GLA area codes:")
print(sorted(gla_harmonised["area_code"].dropna().unique().tolist()))
print()

print("Census area codes:")
print(sorted(census_population_harmonised["area_code"].dropna().unique().tolist()))
print()

print("Fingertips area codes:")
print(sorted(fingertips_harmonised["area_code"].dropna().unique().tolist()))

GLA area codes:
['E09000005', 'E09000009', 'E09000013', 'E09000015', 'E09000017', 'E09000018', 'E09000020', 'E09000033']

Census area codes:
['E09000005', 'E09000009', 'E09000013', 'E09000015', 'E09000017', 'E09000018', 'E09000020', 'E09000033']

Fingertips area codes:
['E09000005', 'E09000009', 'E09000013', 'E09000015', 'E09000017', 'E09000018', 'E09000020', 'E09000033']


All core evidence layers now follow a consistent structure and can be exported for modelling.

We will now validate the final harmonised outputs and save them for the modelling notebook.

In [142]:
# Harmonised table summary
harmonised_summary = pd.DataFrame(
    [
        {
            "dataset_name": "census_population_harmonised",
            "rows": len(census_population_harmonised),
            "columns": len(census_population_harmonised.columns),
            "area_geography_values": sorted(census_population_harmonised["area_geography"].dropna().astype(str).unique().tolist()),
        },
        {
            "dataset_name": "gpps_ethnicity_harmonised",
            "rows": len(gpps_ethnicity_harmonised),
            "columns": len(gpps_ethnicity_harmonised.columns),
            "area_geography_values": sorted(gpps_ethnicity_harmonised["area_geography"].dropna().astype(str).unique().tolist()),
        },
        {
            "dataset_name": "gpps_experience_harmonised",
            "rows": len(gpps_experience_harmonised),
            "columns": len(gpps_experience_harmonised.columns),
            "area_geography_values": sorted(gpps_experience_harmonised["area_geography"].dropna().astype(str).unique().tolist()),
        },
        {
            "dataset_name": "ons_births_harmonised",
            "rows": len(ons_births_harmonised),
            "columns": len(ons_births_harmonised.columns),
            "area_geography_values": sorted(ons_births_harmonised["area_geography"].dropna().astype(str).unique().tolist()),
        },
        {
            "dataset_name": "fingertips_harmonised",
            "rows": len(fingertips_harmonised),
            "columns": len(fingertips_harmonised.columns),
            "area_geography_values": sorted(fingertips_harmonised["area_geography"].dropna().astype(str).unique().tolist()),
        },
        {
            "dataset_name": "gla_harmonised",
            "rows": len(gla_harmonised),
            "columns": len(gla_harmonised.columns),
            "area_geography_values": sorted(gla_harmonised["area_geography"].dropna().astype(str).unique().tolist()),
        },
    ]
)

harmonised_summary

,dataset_name,rows,columns,area_geography_values
0,census_population_harmonised,64,15,[Borough]
1,gpps_ethnicity_harmonised,252,13,[ICS]
2,gpps_experience_harmonised,630,12,[ICS]
3,ons_births_harmonised,995,14,[Country]
4,fingertips_harmonised,5112,12,[Borough]
5,gla_harmonised,888,11,[Borough]


In [143]:
# Export harmonised outputs
EXPORT_FILES = {
    "_harmonised_census_population.parquet": census_population_harmonised,
    "_harmonised_gpps_ethnicity.parquet": gpps_ethnicity_harmonised,
    "_harmonised_gpps_experience.parquet": gpps_experience_harmonised,
    "_harmonised_ons_births.parquet": ons_births_harmonised,
    "_harmonised_fingertips.parquet": fingertips_harmonised,
    "_harmonised_gla.parquet": gla_harmonised,
    "_harmonised_summary.csv": harmonised_summary,
}

for file_name, dataframe in EXPORT_FILES.items():
    output_path = PROJECT_DIR / file_name

    if file_name.endswith(".parquet"):
        dataframe.to_parquet(output_path, index=False)
    elif file_name.endswith(".csv"):
        dataframe.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_harmonised_census_population.parquet
Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_harmonised_gpps_ethnicity.parquet
Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_harmonised_gpps_experience.parquet
Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_harmonised_ons_births.parquet
Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_harmonised_fingertips.parquet
Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_harmonised_gla.parquet
Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_harmon

In [144]:
# Final completion check
for file_name in EXPORT_FILES.keys():
    output_path = PROJECT_DIR / file_name
    print(f"{file_name}: {output_path.exists()}")

_harmonised_census_population.parquet: True
_harmonised_gpps_ethnicity.parquet: True
_harmonised_gpps_experience.parquet: True
_harmonised_ons_births.parquet: True
_harmonised_fingertips.parquet: True
_harmonised_gla.parquet: True
_harmonised_summary.csv: True


All core datasets have been harmonised and exported. They are now ready for inequality modelling.